In [ ]:
import os, json, gc
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import cv2
from diffusers import UNet3DConditionModel, DDPMScheduler
from IPython.display import Video,display


2026-01-21 15:42:35.597602: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769010155.819941      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769010155.883918      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769010156.431055      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769010156.431091      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769010156.431094      24 computation_placer.cc:177] computation placer alr

In [2]:
# WebVid-GIF-600k paths
DATA_ROOT = "/kaggle/input/webvid-gif-600k"
GIF_DIR   = DATA_ROOT
ANN_FILE  = f"{DATA_ROOT}/dataset_600K.tsv"

SAVE_DIR = "/kaggle/working/t2v_quality"
os.makedirs(SAVE_DIR, exist_ok=True)

# Video params (favor motion over sharpness)
FRAME_SIZE = 80
FRAMES = 12      # more frames → better motion learning
FPS = 6

# Latent + text
LATENT_DIM = 8
TEXT_DIM = 512
MAX_LEN = 32

# Training
VAE_EPOCHS = 2   # VAE just needs to reconstruct
DIFF_EPOCHS = 4 # diffusion does main learning
BATCH_SIZE = 1
LR = 1e-4             # slower = more stable

# Diffusion sampling
SAMPLE_STEPS = 40     # faster inference
GUIDANCE = 12.0       # avoid oversharpening artifacts

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cuda


In [3]:
# Read WebVid TSV
pairs = []
with open(ANN_FILE, "r", encoding="utf8") as f:
    for i, line in enumerate(f):
        if i == 0:  # skip header
            continue
        parts = line.strip().split("\t")
        if len(parts) < 2:
            continue
        gif_id = parts[0]
        caption = parts[1]
        pairs.append((gif_id, caption))

print("Total pairs in TSV:", len(pairs))

# Limit for training
pairs = pairs[:30000]   # use 10k for now
print("Using pairs:", len(pairs))

# Build mapping: gif_id -> caption
vid2caps = {vid: [cap] for vid, cap in pairs}


Total pairs in TSV: 599999
Using pairs: 30000


In [4]:
special = ["<pad>", "<bos>", "<eos>", "<unk>"]
word2idx = {t: i for i, t in enumerate(special)}
idx2word = {i: t for t, i in word2idx.items()}

def tok(t):
    # simple cleaning
    t = t.lower()
    t = t.replace(".", "").replace(",", "").replace("!", "").replace("?", "")
    return t.split()

# Build vocab from captions
for caps in vid2caps.values():
    for c in caps:
        for w in tok(c):
            if w not in word2idx:
                idx = len(word2idx)
                word2idx[w] = idx
                idx2word[idx] = w

VOCAB_SIZE = len(word2idx)
print("Vocab size:", VOCAB_SIZE)

def encode(text):
    ids = [word2idx["<bos>"]]
    for w in tok(text):
        ids.append(word2idx.get(w, word2idx["<unk>"]))
        if len(ids) >= MAX_LEN - 1:
            break
    ids.append(word2idx["<eos>"])

    # pad
    while len(ids) < MAX_LEN:
        ids.append(word2idx["<pad>"])

    return np.array(ids, dtype=np.int64)


Vocab size: 29048


In [5]:
class WebVidGIF(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs   # list of (gif_id, caption)

    def read_gif(self, path):
        cap = cv2.VideoCapture(path)
        fs = []
        while len(fs) < FRAMES:
            r, f = cap.read()
            if not r:
                break
            f = cv2.resize(f, (FRAME_SIZE, FRAME_SIZE))
            f = torch.tensor(f).permute(2,0,1).float() / 255.
            fs.append(f)
        cap.release()

        # pad if short
        if len(fs) == 0:
            fs = [torch.zeros(3, FRAME_SIZE, FRAME_SIZE)]
        while len(fs) < FRAMES:
            fs.append(fs[-1])

        return torch.stack(fs)   # [T, C, H, W]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        gif_id, cap = self.pairs[i]
        path = os.path.join(GIF_DIR, gif_id + ".gif")
        v = self.read_gif(path)
        return {
            "video": v,                               # [T, C, H, W]
            "input_ids": torch.tensor(encode(cap)),   # [L]
            "caption": cap
        }

train_loader = DataLoader(
    WebVidGIF(pairs),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [ ]:
class VideoVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv3d(3, 64, 4, 2, 1), nn.ReLU(),      # /2
            nn.Conv3d(64, 128, 4, 2, 1), nn.ReLU(),    # /4
            nn.Conv3d(128, LATENT_DIM, 3, 1, 1)       # same
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose3d(LATENT_DIM, 128, 3, 1, 1), nn.ReLU(),
            nn.ConvTranspose3d(128, 64, 4, 2, 1), nn.ReLU(),   # x2
            nn.ConvTranspose3d(64, 3, 4, 2, 1), nn.Sigmoid()   # x4
        )

    def encode(self, x):   # x: [B,C,T,H,W]
        return self.enc(x)
    def decode(self, z):
        return self.dec(z)
    def forward(self, x):
        return self.decode(self.encode(x))
class TextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(word2idx), TEXT_DIM)
        self.pos = nn.Embedding(MAX_LEN, TEXT_DIM)
        layer = nn.TransformerEncoderLayer(TEXT_DIM, 8, 2048, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, 6)

    def forward(self, ids):
        ids = ids.clamp(0, len(word2idx)-1)   # avoid crashes
        L = ids.size(1)
        pos = torch.arange(L, device=ids.device)
        return self.enc(self.emb(ids) + self.pos(pos))
LATENT_T = FRAMES // 4
LATENT_HW = FRAME_SIZE // 4

unet = UNet3DConditionModel(
    sample_size=(LATENT_T, LATENT_HW, LATENT_HW),
    in_channels=LATENT_DIM,
    out_channels=LATENT_DIM,
    down_block_types=(
        "CrossAttnDownBlock3D",
        "CrossAttnDownBlock3D",
        "DownBlock3D"
    ),
    up_block_types=(
        "UpBlock3D",
        "CrossAttnUpBlock3D",
        "CrossAttnUpBlock3D"
    ),
    block_out_channels=(64, 128, 256),
    cross_attention_dim=TEXT_DIM,
).to(DEVICE)
scheduler = DDPMScheduler(num_train_timesteps=500)


In [7]:
vae = VideoVAE().to(DEVICE)
opt_vae = torch.optim.AdamW(vae.parameters(), lr=LR)

scaler = torch.amp.GradScaler("cuda")
best_vae = 1e9


for e in range(1, VAE_EPOCHS+1):
    tot = 0
    pbar = tqdm(train_loader, desc=f"VAE {e}/{VAE_EPOCHS}")

    for b in pbar:
        v = b["video"].float().to(DEVICE).permute(0,2,1,3,4)  # [B,C,T,H,W]
        opt_vae.zero_grad()

        with torch.amp.autocast("cuda"):
            r = vae(v)

            # Match time dimension safely
            min_t = min(r.shape[2], v.shape[2])
            r2 = r[:, :, :min_t]
            v2 = v[:, :, :min_t]

            loss = F.mse_loss(r2, v2)

        scaler.scale(loss).backward()
        scaler.unscale_(opt_vae)
        torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
        scaler.step(opt_vae)
        scaler.update()

        tot += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.5f}")

    avg = tot / len(train_loader)
    print(f"[VAE] Epoch {e} avg loss = {avg:.6f}")

    if avg < best_vae:
        best_vae = avg
        torch.save(vae.state_dict(), f"{SAVE_DIR}/best_vae.pt")
        print(f"⭐ New best VAE saved at epoch {e} with loss {avg:.6f}")


VAE 1/2:   0%|          | 0/30000 [00:00<?, ?it/s]

[VAE] Epoch 1 avg loss = 0.000260
⭐ New best VAE saved at epoch 1 with loss 0.000260


VAE 2/2:   0%|          | 0/30000 [00:00<?, ?it/s]

[VAE] Epoch 2 avg loss = 0.000000
⭐ New best VAE saved at epoch 2 with loss 0.000000


In [8]:
text_encoder = TextEncoder().to(DEVICE)

opt = torch.optim.AdamW(
    list(unet.parameters()) + list(text_encoder.parameters()),
    lr=LR
)

scaler = torch.amp.GradScaler("cuda")
best_diff = 1e9

for e in range(1, DIFF_EPOCHS+1):
    tot = 0
    pbar = tqdm(train_loader, desc=f"DIFF {e}/{DIFF_EPOCHS}")

    for b in pbar:
        # [B,T,C,H,W] -> [B,C,T,H,W]
        v = b["video"].float().to(DEVICE).permute(0,2,1,3,4)

        ids = b["input_ids"].long().to(DEVICE)
        ids = ids.clamp(0, len(word2idx)-1)

        # Encode with frozen VAE
        with torch.no_grad():
            lat = vae.encode(v)

        B = lat.size(0)
        t = torch.randint(
            0,
            scheduler.config.num_train_timesteps,
            (B,),
            device=DEVICE
        )

        noise = torch.randn_like(lat)
        noisy = scheduler.add_noise(lat, noise, t)

        with torch.amp.autocast("cuda"):
            emb = text_encoder(ids)
            pred = unet(noisy, t, encoder_hidden_states=emb).sample
            loss = F.mse_loss(pred, noise)

        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(
            list(unet.parameters()) + list(text_encoder.parameters()),
            1.0
        )
        scaler.step(opt)
        scaler.update()

        tot += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.5f}")

    avg = tot / len(train_loader)
    print(f"[DIFF] Epoch {e} avg loss = {avg:.6f}")

    if avg < best_diff:
        best_diff = avg
        torch.save(
            {
                "unet": unet.state_dict(),
                "text": text_encoder.state_dict(),
                "epoch": e,
                "loss": avg
            },
            f"{SAVE_DIR}/best_diff.pt"
        )
        print(f"⭐ New best Diffusion saved at epoch {e}, loss {avg:.6f}")

DIFF 1/4:   0%|          | 0/30000 [00:00<?, ?it/s]

[DIFF] Epoch 1 avg loss = 0.017446
⭐ New best Diffusion saved at epoch 1, loss 0.017446


DIFF 2/4:   0%|          | 0/30000 [00:00<?, ?it/s]

[DIFF] Epoch 2 avg loss = 0.004425
⭐ New best Diffusion saved at epoch 2, loss 0.004425


DIFF 3/4:   0%|          | 0/30000 [00:00<?, ?it/s]

[DIFF] Epoch 3 avg loss = 0.003403
⭐ New best Diffusion saved at epoch 3, loss 0.003403


DIFF 4/4:   0%|          | 0/30000 [00:00<?, ?it/s]

[DIFF] Epoch 4 avg loss = 0.002917
⭐ New best Diffusion saved at epoch 4, loss 0.002917


In [9]:
@torch.no_grad()
def generate(prompt):
    ids = torch.tensor(encode(prompt), dtype=torch.long).unsqueeze(0).to(DEVICE)

    txt = text_encoder(ids)
    null_ids = torch.zeros_like(ids)
    null = text_encoder(null_ids)

    LAT_T = FRAMES // 4
    LAT_HW = FRAME_SIZE // 4

    lat = torch.randn(
        1, LATENT_DIM, LAT_T, LAT_HW, LAT_HW,
        device=DEVICE
    )

    scheduler.set_timesteps(SAMPLE_STEPS, device=DEVICE)

    for t in scheduler.timesteps:
        x = torch.cat([lat, lat], dim=0)
        e = torch.cat([null, txt], dim=0)

        p = unet(x, t, encoder_hidden_states=e).sample
        u, c = p.chunk(2)

        lat = scheduler.step(
            u + GUIDANCE * (c - u),
            t,
            lat
        ).prev_sample

    # Decode latent -> video
    vid = vae.decode(lat)   # [1,3,T,H,W]
    return vid
def save_video(t, path):
    t = t.detach().cpu()
    v = t[0]                  # [C,T,H,W]
    v = v.permute(1,2,3,0)    # [T,H,W,C]
    v = (v - v.min()) / (v.max() - v.min() + 1e-8)
    v = (v * 255).byte().numpy()

    T, H, W, C = v.shape
    out = cv2.VideoWriter(
        path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        FPS,
        (W, H)
    )
    for f in v:
        out.write(f)
    out.release()
    print("Saved:", path)
video = generate(" ")
path = "/kaggle/working/generated_quality.mp4"
save_video(video, path)


Saved: /kaggle/working/generated_quality.mp4
